# Beyond the course · Streams into the lakehouse, live

Optional, self-paced: goes deeper than Session 4.3.

**Beyond the course:** no session or lab covers streams into a lakehouse. It combines Session 4.1 (Iceberg) and
Session 4.3 (streaming). **No exercises:** run every cell and look at what happens to the table.

Session 4.3's courier events now land in an Iceberg table every 30 minutes, the way a Spark or Flink job would write them.

1. Micro-batches and offsets
2. The naive sink: commit the data, then save the offset
3. The fix: the offset rides in the same commit
4. Streaming makes small files
5. The table is a stream too
6. Kappa: fix it by replaying the log
7. The same pattern in Spark

🤔 asks you to predict **before** you run the cell.

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/SDAIAAcademy/data-engineering.git"
    if not Path("/content/data-engineering").exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/data-engineering"], check=True)
    os.chdir("/content/data-engineering/labs")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
else:
    while not Path("de_utils.py").exists() and Path.cwd() != Path.cwd().parent:
        os.chdir("..")  # demos/ notebooks run from labs/
sys.path.insert(0, os.getcwd())

import de_utils as U
U.ensure_data()
print("data:", U.RAW, "| quick mode:", U.QUICK)

data: /content/data-engineering/labs\data\raw | quick mode: False


In [2]:
import json
from datetime import datetime, timedelta, timezone

import pandas as pd
import pyarrow as pa
from pyiceberg.transforms import DayTransform

def parse_ts(s: str) -> datetime:
    """The three event_time formats from Session 4.3 -> aware UTC datetime."""
    if s.isdigit():
        return datetime.fromtimestamp(int(s) / 1000, tz=timezone.utc)
    return datetime.fromisoformat(s.replace("Z", "+00:00")).astimezone(timezone.utc)

events = [json.loads(line) for line in open(U.RAW / "events" / "delivery_events.jsonl", encoding="utf-8")]
for e in events:
    e["event_ts"], e["ingest_ts"] = parse_ts(e["event_time"]), parse_ts(e["ingest_time"])

START = datetime(2026, 9, 24, tzinfo=timezone.utc)
stream = [e for e in events if e["ingest_ts"] < START + timedelta(days=2)]   # two days of the stream

SCHEMA = pa.schema([("event_id", pa.string()), ("order_id", pa.int64()), ("store_id", pa.string()), ("city", pa.string()),
                    ("courier_id", pa.string()), ("event_type", pa.string()), ("event_ts", pa.timestamp("us", tz="UTC")),
                    ("ingest_ts", pa.timestamp("us", tz="UTC"))])

def to_arrow(rows: list[dict]) -> pa.Table:
    return pa.Table.from_pylist([{f: r[f] for f in SCHEMA.names} for r in rows], schema=SCHEMA)

DEMO = U.lab_dir("demo_stream_to_lakehouse", reset=True)
catalog = U.get_catalog(DEMO / "lakehouse")
catalog.create_namespace("bronze")

def new_table(name):
    tbl = catalog.create_table(f"bronze.{name}", schema=SCHEMA)
    with tbl.update_spec() as spec:
        spec.add_field("event_ts", DayTransform(), "event_day")
    return tbl

print(f"{len(stream):,} events over two days")

7,483 events over two days


## 1 · Micro-batches and offsets

Spark Structured Streaming works in **micro-batches**: every trigger interval, take whatever arrived, process it, commit
it. The **offset** is the position in the log after the batch: "everything before here is done".

In [3]:
def micro_batches(rows, every=timedelta(minutes=30)):
    """Yield (end_offset, rows, batch_end) per processing-time interval; offsets index the arrival-ordered stream."""
    i, t = 0, START
    while i < len(rows):
        t += every
        j = i
        while j < len(rows) and rows[j]["ingest_ts"] < t:
            j += 1
        if j > i:
            yield j, rows[i:j], t
        i = j

batches = list(micro_batches(stream))
print(len(batches), "micro-batches of 30 minutes")
pd.DataFrame([{"batch_end (UTC)": f"{t:%m-%d %H:%M}", "events": len(rows), "end offset": end}
              for end, rows, t in batches[:6]])

88 micro-batches of 30 minutes


,batch_end (UTC),events,end offset
0,09-24 02:30,3,3
1,09-24 03:00,1,4
2,09-24 03:30,4,8
3,09-24 04:00,17,25
4,09-24 04:30,37,62
5,09-24 05:00,53,115


## 2 · The naive sink: commit the data, then save the offset

A first attempt: append each batch to Iceberg, **then** write the offset to a checkpoint file. On restart, resume
from the checkpoint. The job gets killed once, right after batch 30's data commit.

🤔 After the restart, how many rows does the table have compared with the stream?

In [4]:
class Crash(Exception):
    pass

CKPT = DEMO / "checkpoint.json"

def naive_job(tbl, crash_after=None):
    start = json.loads(CKPT.read_text())["offset"] if CKPT.exists() else 0
    n = 0
    for end, rows, _ in micro_batches(stream):
        if end <= start:
            continue
        tbl.append(to_arrow(rows))                         # 1. commit the data
        n += 1
        if n == crash_after:
            raise Crash(f"💥 killed after batch {n}: data committed, checkpoint not yet written")
        CKPT.write_text(json.dumps({"offset": end}))       # 2. then save the progress

naive = new_table("events_naive")
try:
    naive_job(naive, crash_after=30)
except Crash as err:
    print(err)
print("restart from checkpoint offset", json.loads(CKPT.read_text())["offset"])
naive_job(naive)

rows = naive.scan(selected_fields=("event_id",)).to_arrow().num_rows
print(f"rows in table {rows:,} | events in stream {len(stream):,} | extra {rows - len(stream):,} "
      f"| batch 30 had {len(batches[29][1]):,} events")

💥 killed after batch 30: data committed, checkpoint not yet written
restart from checkpoint offset 2296


rows in table 7,635 | events in stream 7,483 | extra 152 | batch 30 had 152 events


The extra rows are exactly batch 30, written twice. The data and the checkpoint are **two separate writes**; any crash
between them replays a batch. Making the checkpoint write "faster" doesn't help: the gap is still there.

## 3 · The fix: the offset rides in the same commit

Every Iceberg snapshot carries a free-form **summary**. Put the offset there, and one atomic commit saves both the data
and the progress. There is no moment where one exists without the other.

In [5]:
safe = new_table("delivery_events")
end, rows_1, batch_end = batches[0]
safe.append(to_arrow(rows_1), snapshot_properties={"qafila.offset": str(end), "qafila.batch_end": batch_end.isoformat()})

snap = safe.current_snapshot()
{k: v for k, v in snap.summary.additional_properties.items() if k.startswith(("qafila", "added-records"))}

{'added-records': '3',
 'qafila.offset': '3',
 'qafila.batch_end': '2026-09-24T02:30:00+00:00'}

On restart, the job reads the offset back from the table's latest snapshot. That's the same idea as Spark's
checkpoints tied to Iceberg/Delta commit ids, or Flink's two-phase commit. A real sink would
also remove the phones' duplicate events.

## 4 · Streaming makes small files

Back to the naive table: two days of 30-minute batches.

🤔 How many data files, and how big?

In [6]:
files = naive.inspect.files().to_pandas()
files["day"] = files.partition.map(lambda p: p["event_day"])
print(f"snapshots {len(naive.snapshots())} | data files {len(files)} | "
      f"average file {files.file_size_in_bytes.mean() / 1e3:.1f} kB | rows {files.record_count.sum():,}")
files.groupby("day").agg(files=("file_path", "size"), rows=("record_count", "sum"),
                         avg_kB=("file_size_in_bytes", lambda s: round(s.mean() / 1e3, 1)))

snapshots 89 | data files 89 | average file 5.1 kB | rows 7,635


,files,rows,avg_kB
day,,,
2026-09-24,43,3863,5.2
2026-09-25,46,3772,5.0


One file per batch, a few kB each, and that's only two days and one small topic. Each file costs an open, a footer read,
a manifest entry and an object-store request; analytics engines like files of 100 MB or more. Shorter triggers mean
fresher data **and** more, smaller files.

The fix is **compaction**: rewrite each partition's small files into a few large ones, as one more commit. In
production it's Spark's `rewrite_data_files` or a managed service.

## 5 · The table is a stream too

Each micro-batch is one snapshot, and Iceberg remembers what each one **added**.

In [7]:
snaps = naive.inspect.snapshots().to_pandas()
snaps.assign(added_records=snaps.summary.map(lambda s: int(dict(s)["added-records"])))[
    ["committed_at", "snapshot_id", "operation", "added_records"]].tail(5)

,committed_at,snapshot_id,operation,added_records
84,2026-10-30 20:46:28.324,163392503696411111,append,28
85,2026-10-30 20:46:28.512,4608901977672071084,append,13
86,2026-10-30 20:46:28.681,8175929090063526093,append,11
87,2026-10-30 20:46:28.985,1261594312838459180,append,4
88,2026-10-30 20:46:29.147,1593378975662993473,append,7


So a downstream job (bronze → silver) doesn't need to rescan the table. It remembers the last snapshot it processed,
like a consumer offset, and asks for "rows appended since snapshot X". That's how Flink and Spark stream **from**
Iceberg, and how jobs chain through the lakehouse instead of through more Kafka topics.

## 6 · Kappa: fix it by replaying the log

The naive table has a replayed batch and the phones' duplicates in it. In a **Kappa** architecture you don't patch it
with a separate batch job: you fix the code and **replay the retained log** from offset 0 into a new table.

In [8]:
seen, fixed = set(), []
for e in stream:                                   # replay from offset 0 with the fixed logic: one row per event_id
    if e["event_id"] not in seen:
        seen.add(e["event_id"])
        fixed.append(e)

rebuilt = new_table("delivery_events_v2")
rebuilt.append(to_arrow(fixed), snapshot_properties={"qafila.offset": str(len(stream))})
ids = rebuilt.scan(selected_fields=("event_id",)).to_arrow()["event_id"]
print(f"naive: {rows:,} rows | rebuilt: {len(ids):,} rows, {len(set(ids.to_pylist())):,} distinct event ids")

naive: 7,635 rows | rebuilt: 7,384 rows, 7,384 distinct event ids


One code path, and the log is the source of truth. The catch: the log must be retained long enough to replay, and the
sink must be replaceable. The common middle ground in 2026: stream into Iceberg (bronze), then run the **same dbt
models** every 5-15 minutes on the new snapshots.

## 7 · The same pattern in Spark

Shown for reading, not run here:

```python
(events_stream                                            # readStream from Kafka, parsed and deduplicated
    .writeStream.format("iceberg")
    .trigger(processingTime="30 minutes")                 # micro-batches
    .option("checkpointLocation", "s3://…/chk/events")    # offsets, tied to Iceberg commit ids: exactly-once
    .toTable("lake.bronze.delivery_events"))

spark.sql("CALL lake.system.rewrite_data_files('bronze.delivery_events')")    # compaction
spark.sql("CALL lake.system.expire_snapshots('bronze.delivery_events')")      # keep storage bounded
```

## Recap

| We saw | Because |
|---|---|
| one batch written twice | data commit and checkpoint were two separate writes |
| the offset stored in the snapshot | one atomic commit = data + progress |
| one tiny file per micro-batch | every commit writes new files; nothing merges them |
| `added-records` per snapshot | the table can be read incrementally, like a log |
| a clean table from offset 0 | Kappa: replay the retained log with fixed code |

**To try on your own:** a crash-proof sink with dedupe, compaction with before/after timings, a live SLA query on the
table with DuckDB, and incremental reads for the next job.